# LoRA Fine-tuning MARLIN для задачи обнаружения лжи

**Метод:** Low-Rank Adaptation (LoRA) видео-трансформера MARLIN vit_base  
**Датасет:** Real-Life Trial Dataset (119 видео, 54 субъекта, 60 truthful / 59 deceptive)  
**Цель:** Исследовать, улучшает ли доменная адаптация MARLIN через LoRA качество классификации по сравнению с базовым линейным пробингом (balanced accuracy = 0.668)  

**Требования Kaggle:** GPU T4, Internet включён, датасет `katushkastokom/marlin-face-crops` подключён

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q peft marlin-pytorch opencv-python-headless einops scikit-learn tqdm

In [ ]:
import os, types, warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, matthews_corrcoef
from tqdm import tqdm

warnings.filterwarnings("ignore")

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SEED        = 42
N_FOLDS     = 5
LORA_R      = 4
LORA_ALPHA  = 8
LORA_DROP   = 0.1
EPOCHS      = 30
LR          = 3e-4
CLIP_FRAMES = 16
SAMPLE_RATE = 2     # брать каждый 2-й кадр (как в оригинальном MARLIN)
IMG_SIZE    = 224
EMBED_DIM   = 768

INPUT_DIR      = Path("/kaggle/input/datasets/katushkastokom/marlin-face-crops")
FACE_CROPS_DIR = INPUT_DIR / "face_crops-20260411T211821Z-3-001/face_crops"
WORK_DIR       = Path("/kaggle/working")

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Device:", DEVICE)
print("Face crops dir exists:", FACE_CROPS_DIR.exists())
if DEVICE == "cuda":
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 1. Сканирование датасета

In [ ]:
video_exts = {".mp4", ".mov", ".avi", ".mkv"}
rows = []
for vp in sorted(FACE_CROPS_DIR.rglob("*")):
    if vp.suffix.lower() not in video_exts:
        continue
    label_name = "deceptive" if "_lie_" in vp.name else "truthful"
    label      = 0 if label_name == "deceptive" else 1
    subject_id = vp.stem.split("_")[0]
    rows.append({"path": str(vp), "label": label,
                 "label_name": label_name, "subject_id": subject_id})

meta = pd.DataFrame(rows)
print(f"Видео: {len(meta)}  |  субъектов: {meta['subject_id'].nunique()}")
print(meta["label_name"].value_counts().to_string())
meta.head(3)

## 2. Загрузка видео

In [ ]:
def load_video_clips(video_path: str) -> torch.Tensor:
    """Возвращает тензор (N_clips, C, T, H, W) в [0,1]."""
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frames.append(frame)
    cap.release()

    if not frames:
        return None

    frames = frames[::SAMPLE_RATE]              # субдискретизация
    while len(frames) < CLIP_FRAMES:            # дополнение до 16 кадров
        frames.append(frames[-1])

    clips = []
    for start in range(0, len(frames) - CLIP_FRAMES + 1, CLIP_FRAMES):
        c = frames[start : start + CLIP_FRAMES]
        t = torch.from_numpy(
            np.stack(c).transpose(3, 0, 1, 2)  # (C, T, H, W)
        ).float() / 255.0
        clips.append(t)

    if not clips:
        c = frames[:CLIP_FRAMES]
        t = torch.from_numpy(np.stack(c).transpose(3, 0, 1, 2)).float() / 255.0
        clips.append(t)

    return torch.stack(clips)  # (N, C, T, H, W)


# проверка
sample = load_video_clips(meta["path"].iloc[0])
print("Тестовые клипы:", sample.shape, "  dtype:", sample.dtype)

## 3. Построение модели: MARLIN + LoRA

**Почему нужен патч Attention.forward?**  
MARLIN читает вес напрямую: `F.linear(x, self.qkv.weight, bias)`, минуя `self.qkv.forward()`. PEFT-LoRA оборачивает именно `forward()`, поэтому без патча адаптер не применяется. Заменяем `F.linear(...)` на `self.qkv(x)` — теперь LoRA нормально перехватывает вызов.

In [ ]:
from marlin_pytorch import Marlin
from marlin_pytorch.model.modules import Attention
from peft import LoraConfig, get_peft_model


def _patched_attn_forward(self, x):
    """Attention.forward с вызовом self.qkv(x) вместо F.linear — нужно для PEFT LoRA."""
    B, N, C = x.shape
    qkv = self.qkv(x)   # теперь PEFT-адаптер перехватывает этот вызов
    qkv = qkv.reshape(B, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    q = q * self.scale
    attn = (q @ k.transpose(-2, -1)).softmax(dim=-1)
    attn = self.attn_drop(attn)
    x = (attn @ v).transpose(1, 2).reshape(B, N, -1)
    x = self.proj(x)
    x = self.proj_drop(x)
    return x


LORA_CFG = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["qkv"],
    lora_dropout=LORA_DROP,
    bias="none",
)


def build_model():
    """Загружает MARLIN, патчит attention, применяет LoRA, возвращает (encoder, head)."""
    m = Marlin.from_online("marlin_vit_base_ytf")
    enc = m.encoder.to(DEVICE)
    del m  # декодер не нужен

    for module in enc.modules():
        if isinstance(module, Attention):
            module.forward = types.MethodType(_patched_attn_forward, module)

    enc = get_peft_model(enc, LORA_CFG)

    head = nn.Sequential(
        nn.LayerNorm(EMBED_DIM),
        nn.Dropout(0.3),
        nn.Linear(EMBED_DIM, 1),
    ).to(DEVICE)

    return enc, head


# Одноразовая проверка числа параметров (не держим модель в памяти)
_enc_test, _head_test = build_model()
_enc_test.print_trainable_parameters()
print("Параметров головы:", sum(p.numel() for p in _head_test.parameters()))
del _enc_test, _head_test
torch.cuda.empty_cache()
print("Проверка завершена, GPU очищен.")

## 4. Функции обучения и оценки

In [ ]:
def train_epoch(enc, head, optimizer, criterion, df_train, scaler):
    """
    Обучение одной эпохи.
    Ключевое решение для экономии памяти GPU:
    - Используем ОДИН случайный клип на видео (вместо всех клипов).
      Это исключает накопление гигантского вычислительного графа
      (при 5 клипах = 5×1.8 GB активаций одновременно → OOM).
    - Mixed precision (autocast) уменьшает память активаций в ~2 раза.
    - Градиенты накапливаются по всем видео, backward — после каждого.
    """
    enc.train(); head.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    rng = np.random.default_rng()  # новый генератор каждую эпоху

    optimizer.zero_grad()
    df_shuffled = df_train.sample(frac=1).reset_index(drop=True)

    for _, row in df_shuffled.iterrows():
        clips = load_video_clips(row["path"])
        if clips is None:
            continue

        # один случайный клип → маленький граф
        clip = clips[rng.integers(0, len(clips))].unsqueeze(0).to(DEVICE)  # (1,C,T,H,W)
        label = torch.tensor([[float(row["label"])]], device=DEVICE)        # (1,1)

        with torch.cuda.amp.autocast():
            feat  = enc.extract_features(clip, seq_mean_pool=True)  # (1, D)
            logit = head(feat)                                        # (1, 1)
            loss  = criterion(logit, label) / len(df_shuffled)

        scaler.scale(loss).backward()  # граф сразу освобождается
        total_loss += loss.item() * len(df_shuffled)

        with torch.no_grad():
            prob = torch.sigmoid(logit).item()
            all_preds.append(1 if prob >= 0.5 else 0)
            all_labels.append(int(row["label"]))

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
    torch.cuda.empty_cache()

    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    return total_loss / len(df_shuffled), bal_acc


@torch.no_grad()
def eval_fold(enc, head, df_val):
    """
    Оценка на val: используем ВСЕ клипы (без градиентов — память не нужна).
    Агрегация: std по клипам (лучший метод из базовых экспериментов).
    """
    enc.eval(); head.eval()
    probs, preds, labels = [], [], []

    for _, row in df_val.iterrows():
        clips = load_video_clips(row["path"])
        if clips is None:
            continue

        clip_feats = []
        for clip in clips:
            x = clip.unsqueeze(0).to(DEVICE)
            with torch.cuda.amp.autocast():
                f = enc.extract_features(x, seq_mean_pool=True)  # (1, D)
            clip_feats.append(f.float())

        clip_feats = torch.cat(clip_feats, dim=0)  # (N, D)
        feat = clip_feats.std(dim=0, unbiased=False) if len(clip_feats) > 1 \
               else clip_feats.mean(dim=0)

        with torch.cuda.amp.autocast():
            logit = head(feat.unsqueeze(0))
        prob = torch.sigmoid(logit.float()).item()
        probs.append(prob)
        preds.append(1 if prob >= 0.5 else 0)
        labels.append(int(row["label"]))

    bal_acc = balanced_accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = float("nan")
    mcc = matthews_corrcoef(labels, preds)
    return {"bal_acc": bal_acc, "f1": f1, "auc": auc, "mcc": mcc,
            "probs": probs, "labels": labels}


print("Функции обучения определены.")

## 5. Кросс-валидация (subject-level, 5 фолдов)

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X_idx = np.arange(len(meta))
y_arr = meta["label"].to_numpy()
grp   = meta["subject_id"].to_numpy()

fold_results   = []
fold_histories = []

for fold_idx, (tr_idx, te_idx) in enumerate(
    sgkf.split(X_idx, y_arr, groups=grp), start=1
):
    print(f"\n{'='*55}")
    print(f"Fold {fold_idx}/{N_FOLDS}  |  train={len(tr_idx)}  val={len(te_idx)}")

    df_train = meta.iloc[tr_idx].reset_index(drop=True)
    df_val   = meta.iloc[te_idx].reset_index(drop=True)

    # строим свежую модель — GPU свободен (нет глобальных моделей)
    enc, head = build_model()

    n_pos = df_train["label"].sum()
    n_neg = len(df_train) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        [p for p in enc.parameters() if p.requires_grad] + list(head.parameters()),
        lr=LR, weight_decay=1e-2
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-5
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    history  = defaultdict(list)
    best_bal = 0.0
    best_info = {}

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_bal = train_epoch(enc, head, optimizer, criterion, df_train, scaler)
        val_m = eval_fold(enc, head, df_val)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_bal"].append(tr_bal)
        history["val_bal"].append(val_m["bal_acc"])
        history["val_auc"].append(val_m["auc"])

        if val_m["bal_acc"] > best_bal:
            best_bal  = val_m["bal_acc"]
            best_info = {"epoch": epoch, **val_m}

        if epoch % 5 == 0 or epoch == 1:
            print(f"  ep {epoch:3d} | loss {tr_loss:.4f} | "
                  f"tr_bal {tr_bal:.3f} | val_bal {val_m['bal_acc']:.3f} "
                  f"| val_auc {val_m['auc']:.3f}")

    print(f"  ▶ best ep {best_info['epoch']}: "
          f"bal_acc={best_info['bal_acc']:.3f}  "
          f"auc={best_info['auc']:.3f}  "
          f"f1={best_info['f1']:.3f}  "
          f"mcc={best_info['mcc']:.3f}")

    fold_results.append({
        "fold":    fold_idx,
        "bal_acc": best_info["bal_acc"],
        "f1":      best_info["f1"],
        "auc":     best_info["auc"],
        "mcc":     best_info["mcc"],
        "best_ep": best_info["epoch"],
    })
    fold_histories.append(dict(history))

    del enc, head, optimizer, scheduler, criterion, scaler
    torch.cuda.empty_cache()

print("\nКросс-валидация завершена.")

## 6. Результаты

In [ ]:
results_df = pd.DataFrame(fold_results)

print("Результаты по фолдам:")
print(results_df.to_string(index=False))

print("\nСредние метрики:")
for m in ["bal_acc", "f1", "auc", "mcc"]:
    print(f"  {m:8s}: {results_df[m].mean():.4f} ± {results_df[m].std():.4f}")

results_df.to_csv(WORK_DIR / "lora_fold_results.csv", index=False)
print("\nСохранено: lora_fold_results.csv")

In [ ]:
# ── кривые обучения ───────────────────────────────────────────────
fig, axes = plt.subplots(2, N_FOLDS, figsize=(4 * N_FOLDS, 6))
fig.suptitle("LoRA MARLIN — кривые обучения", fontsize=13)

for fi, hist in enumerate(fold_histories):
    ep = range(1, len(hist["train_loss"]) + 1)

    ax = axes[0, fi]
    ax.plot(ep, hist["train_loss"], color="steelblue")
    ax.set_title(f"Fold {fi+1} — Loss"); ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

    ax = axes[1, fi]
    ax.plot(ep, hist["train_bal"], label="train", color="steelblue")
    ax.plot(ep, hist["val_bal"],   label="val",   color="darkorange")
    ax.set_title(f"Fold {fi+1} — Balanced Acc")
    ax.set_xlabel("Epoch"); ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3); ax.set_ylim(0.3, 1.0)

plt.tight_layout()
plt.savefig(WORK_DIR / "lora_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: lora_training_curves.png")

In [ ]:
# ── сравнение с базовыми методами ────────────────────────────────
lora_bal = results_df["bal_acc"].mean()
lora_auc = results_df["auc"].mean()
lora_f1  = results_df["f1"].mean()
lora_mcc = results_df["mcc"].mean()

comparison = pd.DataFrame({
    "Метод":         ["RF + std_only (заморожен)", "MLP на эмбеддингах", "MARLIN + LoRA"],
    "Bal. Accuracy": [0.668, 0.662, round(lora_bal, 3)],
    "AUC":           [0.628, 0.641, round(lora_auc, 3)],
    "F1-macro":      [0.625, 0.618, round(lora_f1,  3)],
    "MCC":           [0.336, 0.325, round(lora_mcc, 3)],
})
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(len(comparison))
for i, (metric, color) in enumerate(zip(
    ["Bal. Accuracy", "AUC", "F1-macro"],
    ["#4C72B0", "#DD8452", "#55A868"]
)):
    ax.bar(x + i * 0.25, comparison[metric], width=0.23, color=color,
           alpha=0.85, label=metric)

ax.set_xticks(x + 0.25); ax.set_xticklabels(comparison["Метод"], fontsize=9)
ax.set_ylim(0.4, 0.85)
ax.set_ylabel("Метрика")
ax.set_title("Сравнение: заморожен MARLIN vs LoRA-адаптация")
ax.axhline(0.5, color="grey", linestyle="--", alpha=0.4)
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / "lora_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

comparison.to_csv(WORK_DIR / "lora_comparison.csv", index=False)
delta = lora_bal - 0.668
print(f"\nLoRA vs базовый: {delta:+.3f} ({('улучшение' if delta>0 else 'без улучшения')})")

## Итоги

| Параметр | Значение |
|---|---|
| LoRA rank / alpha | 4 / 8 |
| Целевые слои | qkv (матрицы внимания) |
| Обучаемых параметров | ~300K из 86M (0.35%) |
| Агрегация при обучении | mean по 1 случайному клипу |
| Агрегация при оценке | std по всем клипам |
| Кросс-валидация | StratifiedGroupKFold(5) — subject-level |

**Интерпретация:**
- Если LoRA улучшает результат — доменная адаптация работает, признаки стали более дискриминативными
- Если результат сопоставим — 119 видео недостаточно для настройки ViT (ожидаемо), но эксперимент ценен как исследование применимости метода
- Высокий разброс по фолдам (ожидаем std ≈ 0.05–0.10) является ограничением датасета, а не метода